# 05. Data Visualization Static

**Aim:** Create bar charts, histograms, box plots, and heatmaps using matplotlib and seaborn.

## Theory

Different plot types answer different questions. Bar charts compare class counts, histograms show value distributions, box plots summarize spread and outliers, and heatmaps visualize correlations between variables. Matplotlib provides flexible low-level plotting, while seaborn adds higher-level statistical graphics with cleaner defaults.

In [ ]:
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFile
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    silhouette_score,
)

ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings('ignore')
os.environ.setdefault('MPLCONFIGDIR', os.path.abspath('../logs/.mplconfig'))
os.makedirs(os.environ['MPLCONFIGDIR'], exist_ok=True)
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

TRAIN_DIR = Path('../data/PlantDoc-Dataset-master/train/')
TEST_DIR = Path('../data/PlantDoc-Dataset-master/test/')
MANIFEST_PATH = Path('../data/dataset_manifest.csv')
CLEAN_MANIFEST_PATH = Path('../data/dataset_manifest_clean.csv')
FEATURES_PATH = Path('../data/image_features.csv')

def scan_split(split_dir: Path, split_name: str) -> pd.DataFrame:
    records = []
    if not split_dir.exists():
        return pd.DataFrame(columns=['image_path', 'class_name', 'split'])
    for class_dir in sorted([p for p in split_dir.iterdir() if p.is_dir()]):
        for image_path in sorted(class_dir.rglob('*')):
            if image_path.is_file():
                records.append({
                    'image_path': str(image_path.as_posix()),
                    'class_name': class_dir.name,
                    'split': split_name,
                })
    return pd.DataFrame(records)


def ensure_manifest(prefer_clean: bool = True) -> pd.DataFrame:
    target = CLEAN_MANIFEST_PATH if prefer_clean and CLEAN_MANIFEST_PATH.exists() else MANIFEST_PATH
    if target.exists():
        return pd.read_csv(target)

    train_df = scan_split(TRAIN_DIR, 'train')
    test_df = scan_split(TEST_DIR, 'test')
    full_df = pd.concat([train_df, test_df], ignore_index=True)
    full_df['class_name'] = full_df['class_name'].astype(str).str.strip()
    full_df = full_df.drop_duplicates(subset=['image_path']).reset_index(drop=True)
    return full_df


def validate_image(image_path: str):
    path = Path(image_path)
    if not path.exists():
        return False, 'missing'
    try:
        with Image.open(path) as img:
            img.verify()
        return True, 'ok'
    except Exception:
        return False, 'corrupt'


def load_rgb_image(image_path: str, size=(224, 224)):
    with Image.open(image_path) as img:
        rgb = img.convert('RGB').resize(size)
        return np.array(rgb)


def extract_rgb_features(image_array: np.ndarray) -> dict:
    channels = image_array.reshape(-1, 3)
    return {
        'mean_r': float(channels[:, 0].mean()),
        'mean_g': float(channels[:, 1].mean()),
        'mean_b': float(channels[:, 2].mean()),
        'std_r': float(channels[:, 0].std()),
        'std_g': float(channels[:, 1].std()),
        'std_b': float(channels[:, 2].std()),
    }


def ensure_features() -> pd.DataFrame:
    if FEATURES_PATH.exists():
        return pd.read_csv(FEATURES_PATH)

    manifest_df = ensure_manifest(prefer_clean=True).copy()
    quality_records = []
    for image_path in manifest_df['image_path']:
        valid, status = validate_image(image_path)
        quality_records.append((valid, status))
    manifest_df[['is_valid_image', 'file_status']] = pd.DataFrame(quality_records, index=manifest_df.index)
    manifest_df = manifest_df[manifest_df['is_valid_image']].copy().reset_index(drop=True)

    feature_rows = []
    for row in manifest_df.itertuples(index=False):
        image_array = load_rgb_image(row.image_path)
        feature_rows.append({
            'image_path': row.image_path,
            'class_name': row.class_name,
            'split': row.split,
            **extract_rgb_features(image_array),
        })

    features_df = pd.DataFrame(feature_rows)
    return features_df


## Code

In [ ]:
manifest_df = ensure_manifest(prefer_clean=True).copy()
features_df = ensure_features().copy()
plot_dir = Path('../logs/viz_exp5')
plot_dir.mkdir(parents=True, exist_ok=True)

class_counts = manifest_df['class_name'].value_counts().sort_values(ascending=False)
plt.figure(figsize=(14, 6))
sns.barplot(x=class_counts.index, y=class_counts.values, palette='viridis')
plt.title('Image Count per Class')
plt.xlabel('Class Name')
plt.ylabel('Image Count')
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig(plot_dir / 'bar_chart_class_counts.png', dpi=150)
plt.show()

random_classes = manifest_df['class_name'].drop_duplicates().sample(n=min(3, manifest_df['class_name'].nunique()), random_state=42).tolist()
channel_map = {'R': 0, 'G': 1, 'B': 2}
channel_colors = {'R': 'red', 'G': 'green', 'B': 'blue'}
fig, axes = plt.subplots(1, len(random_classes), figsize=(18, 5), squeeze=False)
for ax, class_name in zip(axes[0], random_classes):
    sample_paths = manifest_df.loc[manifest_df['class_name'] == class_name, 'image_path'].head(5)
    for channel_name, channel_idx in channel_map.items():
        values = []
        for image_path in sample_paths:
            values.extend(load_rgb_image(image_path).reshape(-1, 3)[:, channel_idx])
        ax.hist(values, bins=30, alpha=0.35, color=channel_colors[channel_name], label=channel_name)
    ax.set_title(class_name)
    ax.set_xlabel('Pixel intensity')
    ax.legend()
plt.tight_layout()
plt.savefig(plot_dir / 'histogram_rgb_random_classes.png', dpi=150)
plt.show()

top_10_classes = features_df['class_name'].value_counts().head(10).index
plt.figure(figsize=(14, 6))
sns.boxplot(data=features_df[features_df['class_name'].isin(top_10_classes)], x='class_name', y='mean_r', palette='Set3')
plt.title('mean_r Distribution for Top 10 Classes')
plt.xlabel('Class Name')
plt.ylabel('mean_r')
plt.xticks(rotation=75)
plt.tight_layout()
plt.savefig(plot_dir / 'boxplot_mean_r_top10.png', dpi=150)
plt.show()

corr_matrix = features_df[['mean_r', 'mean_g', 'mean_b', 'std_r', 'std_g', 'std_b']].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix of RGB Features')
plt.tight_layout()
plt.savefig(plot_dir / 'heatmap_feature_correlation.png', dpi=150)
plt.show()

sample_grid = (manifest_df.groupby('class_name', group_keys=False)
               .apply(lambda group: group.head(1))
               .reset_index(drop=True)
               .head(28))
fig, axes = plt.subplots(4, 7, figsize=(18, 12))
for ax, (_, row) in zip(axes.flat, sample_grid.iterrows()):
    ax.imshow(load_rgb_image(row['image_path']))
    ax.set_title(row['class_name'], fontsize=8)
    ax.axis('off')
for ax in axes.flat[len(sample_grid):]:
    ax.axis('off')
plt.tight_layout()
plt.savefig(plot_dir / 'sample_image_grid.png', dpi=150)
plt.show()

print('Plots saved to:', plot_dir)

## Results & Evaluation

In [ ]:
saved_files = sorted([path.name for path in plot_dir.glob('*.png')])
print('Saved plot files:')
print(saved_files)
print('Total plots saved:', len(saved_files))

## Conclusion

Static visualizations make it easier to compare class sizes, inspect RGB distributions, identify outliers, and understand feature correlations. The saved figures can also be reused in reports or lab documentation.